In [ ]:
import pickle
from collections import Counter, defaultdict
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from tqdm import tqdm
import networkx as nx
import torch
import numpy as np
from glob import glob

from atomica.data.dataset import BlockGeoAffDataset, PDBDataset
from atomica.models.classifier_model import MultiClassClassifierModel, ClassifierModel
from atomica.data.process_pdbs import process_PL_pdb
from atomica.trainers.abs_trainer import Trainer
from sklearn.metrics import precision_recall_curve, auc
from atomica.data.dataset import VOCAB

In [2]:
# for any missing fastas rewrite them
from Bio import PDB
from Bio.SeqUtils import seq1
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio import SeqIO
from Bio.SeqIO.FastaIO import FastaWriter

def pdb_to_fasta(pdb_filename):
    structure_id = os.path.basename(pdb_filename).split('.')[0]

    # Create a PDB parser
    if pdb_filename.endswith('.cif'):
        parser = PDB.MMCIFParser(QUIET=True)
    else:
        parser = PDB.PDBParser(QUIET=True)

    # Parse the structure
    structure = parser.get_structure("protein_structure", pdb_filename)

    output = {}

    # Open the FASTA file for writing
    for model in structure:
        for chain in model:
            sequence = ""
            for residue in chain:
                # Ensure the residue is an amino acid
                if PDB.is_aa(residue, standard=True):
                    sequence += seq1(residue.get_resname())
            
            # Create a sequence record for the chain
            seq_record = SeqRecord(
                Seq(sequence),
                id=f"{structure_id}_{chain.id}",
                description=f"Chain {chain.id}"
            )

            output[seq_record.id] = str(seq_record.seq)
    return output

In [ ]:
cluster_assignments = pd.read_csv('/n/holylfs06/LABS/mzitnik_lab/Lab/afang/protein_universe/1-AFDBClusters-entryId_repId_taxId.tsv.gz', compression='gzip', sep='\t', names=['memberID', 'repId', 'taxId'])
taxonomy_df = pd.read_csv('/n/holylfs06/LABS/mzitnik_lab/Lab/afang/protein_universe/taxonomy_all_2025_01_15.tsv', sep='\t', dtype={'Taxon Id':int, 'Common name': str, 'Scientific name': str, 'Lineage': str, 'Links': str})
cluster_assignments = cluster_assignments.merge(taxonomy_df[['Taxon Id', 'Lineage']], left_on='taxId', right_on='Taxon Id')

In [ ]:
cluster_assignments['Ancestor'] = cluster_assignments['Lineage'].apply(
    lambda x: x.split(',')[1] if pd.notna(x) and len(x.split(',')) > 1 else x
)

564         NaN
747         NaN
750         NaN
869         NaN
885         NaN
           ... 
30043890    NaN
30043915    NaN
30043916    NaN
30043917    NaN
30044606    NaN
Name: Lineage, Length: 94407, dtype: object

In [66]:
cluster_assignments.to_csv('/n/holylfs06/LABS/mzitnik_lab/Lab/afang/protein_universe/1-AFDBClusters-entryId_repId_taxId_with_taxName.tsv.gz', compression='gzip', sep='\t', index=False)

In [63]:
cluster_assignments['Ancestor']

0            Bacteria
1            Bacteria
2            Bacteria
3            Bacteria
4            Bacteria
              ...    
30045242     Bacteria
30045243     Bacteria
30045244     Bacteria
30045245     Bacteria
30045246     Bacteria
Name: Ancestor, Length: 30045247, dtype: object

## Cluster A0A6B2YTG2, HEM binder

In [11]:
rep_id = 'A0A6B2YTG2'
members = cluster_assignments[cluster_assignments['repId'] == rep_id]['memberID'].tolist()
with open(f"/n/netscratch/mzitnik_lab/Lab/afang/InteractNN/protein_universe/foldseek_cluster/{rep_id}_cluster/{rep_id}_cluster.txt", "w") as f:
    for member in members:
        f.write(member + '\n')
print("Num members:", len(members))

Num members: 134


In [14]:
cluster_assignments[cluster_assignments['repId'] == rep_id]['taxId'].value_counts()

taxId
2053541    3
1898104    3
1978231    3
2026791    2
1904441    2
          ..
2023191    1
1898206    1
1979207    1
2026743    1
2303330    1
Name: count, Length: 124, dtype: int64

In [28]:
with open("/n/netscratch/mzitnik_lab/Lab/afang/InteractNN/protein_universe/foldseek_cluster/A0A6B2YTG2_cluster/processed/A0A6B2YTG2_cluster_80_ligand.pkl", "rb") as f:
    dark_proteome_ligand = pickle.load(f)
len(dark_proteome_ligand)

97

In [ ]:
parameter_tuned_versions = {
    "ADP": "/n/netscratch/mzitnik_lab/Lab/afang/InteractNN/models/Pion_tune/version_325/",
    "ATP": "/n/netscratch/mzitnik_lab/Lab/afang/InteractNN/models/Pion_tune/version_407/",
    "GTP": "/n/netscratch/mzitnik_lab/Lab/afang/InteractNN/models/Pion_tune/version_416/",
    "GDP": "/n/netscratch/mzitnik_lab/Lab/afang/InteractNN/models/Pion_tune/version_397/",
    "FAD": "/n/netscratch/mzitnik_lab/Lab/afang/InteractNN/models/Pion_tune/version_341/",
    "NAD": "/n/netscratch/mzitnik_lab/Lab/afang/InteractNN/models/Pion_tune/version_328/",
    "NAP": "/n/netscratch/mzitnik_lab/Lab/afang/InteractNN/models/Pion_tune/version_344/",
    "NDP": "/n/netscratch/mzitnik_lab/Lab/afang/InteractNN/models/Pion_tune/version_423/",
    "HEM": "/n/netscratch/mzitnik_lab/Lab/afang/InteractNN/models/Pion_tune/version_386/",
    "HEC": "/n/netscratch/mzitnik_lab/Lab/afang/InteractNN/models/Pion_tune/version_362/",
    "CIT": "/n/netscratch/mzitnik_lab/Lab/afang/InteractNN/models/Pion_tune/version_347/",
    "CLA": "/n/netscratch/mzitnik_lab/Lab/afang/InteractNN/models/Pion_tune/version_428/",
}

dark_proteome_predictions = {}

for ligand, version in parameter_tuned_versions.items():
    topk_file = f"{version}/checkpoint/topk_map.txt"
    with open(topk_file, 'r') as f:
        topk_map = f.readlines()
    best_ckpt = topk_map[0].split()[1]

    pion_model = torch.load(best_ckpt, map_location='cuda')
    batch_size = 16
    
    predicted_ligands = []
    for i in range(0, len(dark_proteome_ligand), batch_size):
        end = min(i+batch_size, len(dark_proteome_ligand))
        batch = PDBDataset.collate_fn([dark_proteome_ligand[j]['data'] for j in range(i, end)])
        batch = Trainer.to_device(batch, 'cuda')
        output = pion_model.infer(batch).detach().cpu()
        predicted_ligands.append(output)
    predicted_ligands = torch.cat(predicted_ligands, dim=0).squeeze()
    dark_proteome_predictions[ligand] = predicted_ligands
    print(f"{ligand} done")

ADP done
ATP done
GTP done
GDP done
FAD done
NAD done
NAP done
NDP done
HEM done
HEC done
CIT done
CLA done


In [30]:
dark_proteome_ligand_ids = [x['id'] for x in dark_proteome_ligand]
dark_proteome_predictions_ids = {}
for ligand in parameter_tuned_versions.keys():
    if ligand not in dark_proteome_predictions:
        continue
    dark_proteome_predictions_ids[ligand] = []
    for i, pred in enumerate(dark_proteome_predictions[ligand]):
        if pred > 0.5:
            dark_proteome_predictions_ids[ligand].append(dark_proteome_ligand_ids[i])
    print(ligand, len(dark_proteome_predictions_ids[ligand]), len(dark_proteome_ligand))

ADP 0 97
ATP 0 97
GTP 0 97
GDP 1 97
FAD 0 97
NAD 0 97
NAP 0 97
NDP 0 97
HEM 18 97
HEC 0 97
CIT 0 97
CLA 0 97


In [53]:
cluster_assignments[cluster_assignments['memberID'].isin(dark_proteome_predictions_ids['HEM'])]

,memberID,repId,taxId,Taxon Id,Lineage
19152284,A0A1M2YTW8,A0A6B2YTG2,1895815,1895815,"cellular organisms, Bacteria, Pseudomonadota, ..."
19152291,A0A257ETP5,A0A6B2YTG2,2015578,2015578,"cellular organisms, Bacteria, Pseudomonadota, ..."
19152297,A0A2E8PWR0,A0A6B2YTG2,1898206,1898206,"cellular organisms, Bacteria, Spirochaetota, S..."
19152299,A0A2N2TT14,A0A6B2YTG2,2013707,2013707,"cellular organisms, Bacteria, Pseudomonadota, ..."
19152305,A0A315BDD0,A0A6B2YTG2,1100726,1100726,"cellular organisms, Bacteria, Pseudomonadota, ..."
19152306,A0A315CN29,A0A6B2YTG2,1835767,1835767,"cellular organisms, Bacteria, Pseudomonadota, ..."
19152309,A0A350AHA4,A0A6B2YTG2,1904441,1904441,"cellular organisms, Bacteria, Pseudomonadota, ..."
19152320,A0A3M0CJT2,A0A6B2YTG2,911205,911205,"cellular organisms, Bacteria, Pseudomonadota, ..."
19152330,A0A4Q2UIU3,A0A6B2YTG2,2502893,2502893,"cellular organisms, Bacteria, FCB group, Bacte..."
19152336,A0A4V3CSL8,A0A6B2YTG2,706186,706186,"cellular organisms, Bacteria, FCB group, Bacte..."


In [ ]:
id_to_fasta = {}
all_ids = [x['id'] for x in dark_proteome_ligand]
for id in tqdm(all_ids, total=len(all_ids)):
    if id not in id_to_fasta:
        new_fasta = pdb_to_fasta(f"/n/netscratch/mzitnik_lab/Lab/afang/InteractNN/protein_universe/foldseek_cluster/A0A6B2YTG2_cluster/{id}.pdb")
        id_to_fasta[id] = new_fasta[f'{id}_A']

100%|██████████| 97/97 [00:03<00:00, 26.74it/s]


In [ ]:
import pandas as pd
import json
import shutil
import random
random.seed(42)

ligand='HEM'
out_dir = f"/n/netscratch/mzitnik_lab/Lab/afang/AF3-output/MLP_1_vs_other_{ligand}_{rep_id}_cluster/"
os.makedirs(f"{out_dir}/outputs/model", exist_ok=True)

num_jobs = len(dark_proteome_predictions_ids[ligand])
num_chunks = max(num_jobs//100, 1)
chunk_size = num_jobs//num_chunks
chunks = range(0, num_jobs+chunk_size, chunk_size)
chunks = list(zip(chunks[:-1], chunks[1:]))
chunks[-1] = (chunks[-1][0], num_jobs)

# write AF3 format
jobs = []
for chunk_i, (start, end) in enumerate(chunks):
    os.makedirs(f"{out_dir}/inputs/model{chunk_i}", exist_ok=True)
    for idx in range(start, end):
        item_id = dark_proteome_predictions_ids[ligand][idx]
        if ligand == 'HEC':
            smiles = 'CC=C1C(=C2C=C3C(=CC)C(=C4N3[Fe]56N2C1=Cc7n5c(c(c7C)CCC(=O)O)C=C8N6C(=C4)C(=C8CCC(=O)O)C)C)C'
            item_job1 = {
                "name": item_id + '_model',
                "modelSeeds": [1],
                "sequences": [
                    {"protein": {"id": 'A', "sequence": id_to_fasta[item_id]}},
                    {"ligand": {"id": 'B', "smiles": smiles}},
                ],
                "dialect": "alphafold3",
                "version": 1
            }
        else:
            item_job1 = {
                "name": item_id + '_model',
                "modelSeeds": [1],
                "sequences": [
                    {"protein": {"id": 'A', "sequence": id_to_fasta[item_id]}},
                    {"ligand": {"id": ['B'], "ccdCodes": [ligand]}},
                ],
                "dialect": "alphafold3",
                "version": 1
            }

        with open(f'{out_dir}/inputs/model{chunk_i}/{item_id}_model.json', 'w') as f:
            json.dump(item_job1, f, indent=2)

In [8]:
# analyse AF3 output
from glob import glob
import json
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

rep_id = 'A0A6B2YTG2'
ligand = 'HEM'

results_df = []
model_outputs = glob(f"/n/netscratch/mzitnik_lab/Lab/afang/AF3-output/MLP_1_vs_other_{ligand}_{rep_id}_cluster/outputs/model/**/*summary_confidences.json")

for fname in model_outputs:
    with open(fname, 'r') as f:
        data = json.load(f)
    id_name = os.path.basename(fname).replace("_summary_confidences.json", "")
    iptm = data['iptm']
    ptm = data['ptm']
    with open(fname.replace("_summary_confidences.json", "_data.json"), 'r') as f:
        data = json.load(f)
    if 'ccdCodes' in data['sequences'][1]['ligand']:
        ligand = data['sequences'][1]['ligand']['ccdCodes'][0]
    else:
        ligand = data['sequences'][1]['ligand']['smiles']
        if ligand == 'CC=C1C(=C2C=C3C(=CC)C(=C4N3[Fe]56N2C1=Cc7n5c(c(c7C)CCC(=O)O)C=C8N6C(=C4)C(=C8CCC(=O)O)C)C)C':
            ligand = 'HEC'
    results_df.append({"ligand": ligand, "id": id_name.split("_")[0], "type": id_name.split("_")[1], "iptm": iptm, "ptm": ptm, "ligand": ligand})
results_df = pd.DataFrame(results_df)

In [13]:
results_df[['iptm', 'ptm']].agg(['mean', 'std'])

,iptm,ptm
mean,0.887222,0.892778
std,0.030641,0.023466


compare this HEM binding site to all existing HEM binding sites

In [7]:
test_path = "/n/holylfs06/LABS/mzitnik_lab/Lab/afang/QBioLiP/sequence_30_split/PL_test.pkl"
train_path = "/n/holylfs06/LABS/mzitnik_lab/Lab/afang/QBioLiP/sequence_30_split/PL_train.pkl"
valid_path = "/n/holylfs06/LABS/mzitnik_lab/Lab/afang/QBioLiP/sequence_30_split/PL_valid.pkl"

with open(test_path, 'rb') as f:
    test = pickle.load(f)

with open(train_path, 'rb') as f:
    train = pickle.load(f)

with open(valid_path, 'rb') as f:
    valid = pickle.load(f)


HEM_items = []
for item in train + valid + test:
    if item['id'].split("_")[-2] == 'HEM':
        HEM_items.append(item)

with open("/n/netscratch/mzitnik_lab/Lab/afang/InteractNN/protein_universe/foldseek_cluster/A0A6B2YTG2_cluster/HEM_embeddings/HEM_PDB.pkl", "wb") as f:
    pickle.dump(HEM_items, f)

In [ ]:
rep_id = 'A0A6B2YTG2'
ligand = 'HEM'

cluster_items = []
model_outputs = glob(f"/n/netscratch/mzitnik_lab/Lab/afang/AF3-output/MLP_1_vs_other_{ligand}_{rep_id}_cluster/outputs/model/**/*_model_model.cif")
for cif_file in model_outputs:
    uniprot_id = os.path.basename(cif_file).split("_")[0]
    new_item = process_PL_pdb(cif_file, uniprot_id, "A", "HEM", "B", "Cc1c2n3c(c1CCC(=O)O)C=C4C(=C(C5=[N]4[Fe]36[N]7=C(C=C8N6C(=C5)C(=C8C)C=C)C(=C(C7=C2)C)C=C)C)CCC(=O)O", dist_th=8, fragmentation_method='PS_300')[0]
    cluster_items.append(new_item)

print(f"Processed {len(cluster_items)} out of {len(model_outputs)}")

with open(f"/n/netscratch/mzitnik_lab/Lab/afang/InteractNN/protein_universe/foldseek_cluster/A0A6B2YTG2_cluster/HEM_embeddings/HEM_{rep_id}.pkl", "wb") as f:
    pickle.dump(cluster_items, f)

Could not fragment ligand HEM from /n/netscratch/mzitnik_lab/Lab/afang/AF3-output/MLP_1_vs_other_HEM_A0A6B2YTG2_cluster/outputs/model/a0a1m2ytw8_model/a0a1m2ytw8_model_model.cif. Error=rdkit.Chem.rdchem.BondType.DATIVE
Could not fragment ligand HEM from /n/netscratch/mzitnik_lab/Lab/afang/AF3-output/MLP_1_vs_other_HEM_A0A6B2YTG2_cluster/outputs/model/a0a257etp5_model/a0a257etp5_model_model.cif. Error=rdkit.Chem.rdchem.BondType.DATIVE
Could not fragment ligand HEM from /n/netscratch/mzitnik_lab/Lab/afang/AF3-output/MLP_1_vs_other_HEM_A0A6B2YTG2_cluster/outputs/model/a0a2e8pwr0_model/a0a2e8pwr0_model_model.cif. Error=rdkit.Chem.rdchem.BondType.DATIVE
Could not fragment ligand HEM from /n/netscratch/mzitnik_lab/Lab/afang/AF3-output/MLP_1_vs_other_HEM_A0A6B2YTG2_cluster/outputs/model/a0a2n2tt14_model/a0a2n2tt14_model_model.cif. Error=rdkit.Chem.rdchem.BondType.DATIVE
Could not fragment ligand HEM from /n/netscratch/mzitnik_lab/Lab/afang/AF3-output/MLP_1_vs_other_HEM_A0A6B2YTG2_cluster/outp

In [33]:
with open("/n/netscratch/mzitnik_lab/Lab/afang/InteractNN/protein_universe/foldseek_cluster/A0A6B2YTG2_cluster/HEM_embeddings/HEM_A0A6B2YTG2_embeddings.pkl", "rb") as f:
    cluster_embeddings = pickle.load(f)
with open("/n/netscratch/mzitnik_lab/Lab/afang/InteractNN/protein_universe/foldseek_cluster/A0A6B2YTG2_cluster/HEM_embeddings/HEM_PDB_embeddings.pkl", "rb") as f:
    pdb_embeddings = pickle.load(f)

cluster_graph_embeddings = np.array([x['graph_embedding'] for x in cluster_embeddings])
pdb_graph_embeddings = np.array([x['graph_embedding'] for x in pdb_embeddings])

In [ ]:
cluster_graph_embeddings['id']

In [34]:
from sklearn.metrics.pairwise import cosine_similarity

# Calculate the cosine similarity matrix
cosine_sim_matrix = cosine_similarity(cluster_graph_embeddings, pdb_graph_embeddings)

# Print the shape of the resulting matrix
print(cosine_sim_matrix.shape)

(18, 2610)


In [53]:
self_sim_matrix = cosine_similarity(cluster_graph_embeddings, cluster_graph_embeddings)
self_sim_matrix.mean(), cosine_sim_matrix.mean()

(0.9901086, 0.917388)

In [50]:
for idx in range(len(cluster_embeddings)):
    neighbours = cosine_sim_matrix[idx].argsort()[-10:][::-1]
    neighbour_ids = [pdb_embeddings[neigh]['id'] for neigh in neighbours]
    print(idx, cluster_embeddings[idx]['id'], " ".join(neighbour_ids))

0 a0a1m2ytw8_A_B_HEM 1l0l_1.pdb_1l0l_1_HEM_L.pdb 3dsj_2.pdb_3dsj_2_HEM_H.pdb 1a4e_1.pdb_1a4e_1_HEM_K.pdb 2isa_1.pdb_2isa_1_HEM_I.pdb 6hwh_1.pdb_6hwh_1_HEM_AB.pdb 6hwh_2.pdb_6hwh_2_HEM_AB.pdb 7qan_1.pdb_7qan_1_HEM_C.pdb 1a4e_1.pdb_1a4e_1_HEM_N.pdb 2isa_1.pdb_2isa_1_HEM_Q.pdb 5gux_2.pdb_5gux_2_HEM_F.pdb
1 a0a257etp5_A_B_HEM 1io8_3.pdb_1io8_3_HEM_D3.pdb 3ejd_2.pdb_3ejd_2_HEM_P.pdb 4g1v_2.pdb_4g1v_2_HEM_B4.pdb 4g1v_4.pdb_4g1v_4_HEM_B4.pdb 4iam_1.pdb_4iam_1_HEM_C2.pdb 4g1v_2.pdb_4g1v_2_HEM_B3.pdb 4g1v_4.pdb_4g1v_4_HEM_B1.pdb 4g1v_2.pdb_4g1v_2_HEM_B1.pdb 4g1v_2.pdb_4g1v_2_HEM_B6.pdb 5nws_1.pdb_5nws_1_HEM_E.pdb
2 a0a2e8pwr0_A_B_HEM 2aa1_1.pdb_2aa1_1_HEM_G.pdb 1a4e_1.pdb_1a4e_1_HEM_I.pdb 7rh6_1.pdb_7rh6_1_HEM_SB.pdb 2oyy_2.pdb_2oyy_2_HEM_P.pdb 4e37_1.pdb_4e37_1_HEM_K.pdb 3aq6_1.pdb_3aq6_1_HEM_C.pdb 6a2j_1.pdb_6a2j_1_HEM_B.pdb 6j88_2.pdb_6j88_2_HEM_E.pdb 3r9c_1.pdb_3r9c_1_HEM_B.pdb 6o0a_1.pdb_6o0a_1_HEM_C.pdb
3 a0a2n2tt14_A_B_HEM 5a13_1.pdb_5a13_1_HEM_PA.pdb 2vxh_1.pdb_2vxh_1_HEM_J.pdb 3mk7_3.p

sequence alignment between the HEM binding sites in the cluster

In [108]:
rep_id = 'A0A6B2YTG2'
ligand = 'HEM'
structures = glob(f"/n/netscratch/mzitnik_lab/Lab/afang/AF3-output/MLP_1_vs_other_{ligand}_{rep_id}_cluster/outputs/model/**/*_model_model.cif")
structure_fastas = {}
for structure in structures:
    structure_id = os.path.basename(structure).replace("_model_model.cif", "")
    structure_fastas[structure_id] = pdb_to_fasta(structure)[f'{structure_id}_model_model_A']
    
for item in cluster_items:
    fasta_str = structure_fastas[item['id'].split("_")[0]]
    fe_coords = item['data']['X'][item['data']['A'].index(28)]
    his_coords_matrix = []
    his_blocks = []
    for key, val in item['block_to_pdb_indexes'].items():
        aa_idx = int(val.split("_")[1]) - 1
        if fasta_str[aa_idx] == 'H':
            block_start = sum(item['data']['block_lengths'][:key])
            block_end = block_start + item['data']['block_lengths'][key]
            his_coords = np.array(item['data']['X'])[block_start:block_end].mean(axis=0)
            his_blocks.append(val)
            his_coords_matrix.append(his_coords)
    
    his_coords_matrix = np.array(his_coords_matrix)
    pairwise_distances = np.linalg.norm(his_coords_matrix - fe_coords, axis=1)
    print(item['id'].split("_")[0].upper(), his_blocks[pairwise_distances.argmin()])

A0A1M2YTW8 A_231
A0A257ETP5 A_214
A0A2E8PWR0 A_219
A0A2N2TT14 A_209
A0A315BDD0 A_115
A0A315CN29 A_210
A0A350AHA4 A_214
A0A3M0CJT2 A_205
A0A4Q2UIU3 A_204
A0A4V3CSL8 A_203
A0A554U5W1 A_223
A0A5C1WII0 A_227
A0A6B2YTG2 A_206
A0A6G8IHU8 A_209
A0A7C2HWR6 A_219
A0A7J5TTN5 A_203
A0A7X1LEY2 A_225
A0A839AMV3 A_206


In [110]:
# 1 indexing, determined using PLIP / look for nearest HIS in the structure to the Fe
his_binding_sites = {
    'A0A1M2YTW8': 231,
    'A0A257ETP5': 214,
    'A0A2E8PWR0': 219,
    'A0A2N2TT14': 209,
    'A0A315BDD0': 115,
    'A0A315CN29': 210,
    'A0A350AHA4': 214,
    'A0A3M0CJT2': 205,
    'A0A4Q2UIU3': 204,
    'A0A4V3CSL8': 203,
    'A0A554U5W1': 223,
    'A0A5C1WII0': 227,
    'A0A6B2YTG2': 206,
    'A0A6G8IHU8': 209,
    'A0A7C2HWR6': 219,
    'A0A7J5TTN5': 203,
    'A0A7X1LEY2': 225,
    'A0A839AMV3': 206,
}
for structure_id, his_site in his_binding_sites.items():
    print(structure_fastas[structure_id.lower()][his_site-7:his_site+5])

GDLRVAHLFATH
GDLRIAHFFGTH
GDIRAVHFFGLH
GDLRPAHFIGIH
ARGEGSHYNTTD
GDLRVAHFFALH
GDLRVAHFLATH
GDLRAAHFFATH
GDLRIAHFMGMH
GDLRIAHFIGMH
GDLRVPHFLGIH
GDLRVAHFLGLH
APLKPLHGVSLH
GDLRVPHFFATH
GDLRVAHFFATH
GDLRIAHFLGMH
GDLRVPHFFATH
GDLRVAHFFGLH


search for NN in lipids - does not come back with anything interesting

In [62]:
with open("/n/netscratch/mzitnik_lab/Lab/afang/InteractNN/protein_universe/function/embeddings/prot_interface_version_0_epoch11_step68820/proteins_with_evidence_go_annotation_PeSTo_80_lipid.pkl", "rb") as f:
    lipid_proteins_evidence = pickle.load(f)

with open("/n/netscratch/mzitnik_lab/Lab/afang/InteractNN/protein_universe/function/embeddings/prot_interface_version_0_epoch11_step68820/annotated_uniprot_PeSTo_80_lipid.pkl", "rb") as f:
    lipid_proteins_evidence2 = pickle.load(f)

with open("/n/netscratch/mzitnik_lab/Lab/afang/InteractNN/protein_universe/function/embeddings/prot_interface_version_0_epoch11_step68820/is_dark_90_plddt_PeSTo_80_lipid.pkl", "rb") as f:
    lipid_proteins_dark = pickle.load(f)

lipid_proteins_dark_ids = [x['id'] for x in lipid_proteins_dark]
query_idx = lipid_proteins_dark_ids.index(rep_id)
query_lipid_embedding = lipid_proteins_dark[query_idx]['graph_embedding']

lipid_proteins_embeddings = np.array([x['graph_embedding'] for x in lipid_proteins_evidence])
lipid_proteins_embeddings2 = np.array([x['graph_embedding'] for x in lipid_proteins_evidence2])
lipid_protein_embeddings_all = np.concatenate([lipid_proteins_embeddings, lipid_proteins_embeddings2])
lipid_protein_embeddings_ids = [x['id'] for x in lipid_proteins_evidence] + [x['id'] for x in lipid_proteins_evidence2]

cosine_sim_matrix = cosine_similarity(query_lipid_embedding.reshape(1, -1), lipid_protein_embeddings_all)
for neighbour in cosine_sim_matrix.argsort()[0][-10:][::-1]:
    print(lipid_protein_embeddings_ids[neighbour])

Q9FFD0
A0A521ZRA9
Q9VJA5
Q01650
Q9NZ01
Q3E8X3
Q93084
A0A4P5YKZ2
Q1L864
Q91Y77


## Pion clusters
to check out further: A0A496MS18-CA, A0A352V9P3-ZN, W2EKU0-MG/CA, A0A0Q8TPY4-ZN, A0A552UZ29-ZN

In [2]:
rep_ids = {'A0A496MS18': ['CA'], 'A0A352V9P3': ['ZN'], 'W2EKU0': ['MG', 'CA'], 'A0A0Q8TPY4': ['ZN'], 'A0A552UZ29': ['ZN']}

In [ ]:
# rep_ids = ['A0A496MS18','A0A352V9P3','W2EKU0','A0A0Q8TPY4','A0A552UZ29']

# for rep_id in rep_ids:
#     members = cluster_assignments[cluster_assignments['repId'] == rep_id]['memberID'].tolist()
#     os.makedirs(f"/n/netscratch/mzitnik_lab/Lab/afang/InteractNN/protein_universe/foldseek_cluster/{rep_id}_cluster/", exist_ok=True)
#     with open(f"/n/netscratch/mzitnik_lab/Lab/afang/InteractNN/protein_universe/foldseek_cluster/{rep_id}_cluster/{rep_id}_cluster.txt", "w") as f:
#         for member in members:
#             f.write(member + '\n')
#     print(rep_id, "Num members:", len(members))

NameError: name 'cluster_assignments' is not defined

In [ ]:
parameter_tuned_versions = {
    "CA": "/n/netscratch/mzitnik_lab/Lab/afang/InteractNN/models/Pion_tune/version_305/",
    "CO": "/n/netscratch/mzitnik_lab/Lab/afang/InteractNN/models/Pion_tune/version_304/",
    "CU": "/n/netscratch/mzitnik_lab/Lab/afang/InteractNN/models/Pion_tune/version_317/",
    "FE": "/n/netscratch/mzitnik_lab/Lab/afang/InteractNN/models/Pion_tune/version_314/",
    "K": "/n/netscratch/mzitnik_lab/Lab/afang/InteractNN/models/Pion_tune/version_206/",
    "MG": "/n/netscratch/mzitnik_lab/Lab/afang/InteractNN/models/Pion_tune/version_268/",
    "MN": "/n/netscratch/mzitnik_lab/Lab/afang/InteractNN/models/Pion_tune/version_316/",
    "NA": "/n/netscratch/mzitnik_lab/Lab/afang/InteractNN/models/Pion_tune/version_203/",
    "ZN": "/n/netscratch/mzitnik_lab/Lab/afang/InteractNN/models/Pion_tune/version_209/",
}

prediction_counts = []
rep_ids_predictions = defaultdict(list)
for rep_id, chosen_ions in rep_ids.items():

    with open(f"/n/netscratch/mzitnik_lab/Lab/afang/InteractNN/protein_universe/foldseek_cluster/{rep_id}_cluster/processed/pesto_80_ion.pkl", "rb") as f:
        dark_proteome_ion = pickle.load(f)
    print(f"{rep_id} num Pesto predicted ion binding sites {len(dark_proteome_ion)}")

    dark_proteome_predictions = {}

    for ion, version in parameter_tuned_versions.items():
        topk_file = f"{version}/checkpoint/topk_map.txt"
        with open(topk_file, 'r') as f:
            topk_map = f.readlines()
        best_ckpt = topk_map[0].split()[1]

        pion_model = torch.load(best_ckpt, map_location='cuda')
        batch_size = 16
        
        predicted_ions = []
        for i in range(0, len(dark_proteome_ion), batch_size):
            end = min(i+batch_size, len(dark_proteome_ion))
            batch = PDBDataset.collate_fn([dark_proteome_ion[j]['data'] for j in range(i, end)])
            batch = Trainer.to_device(batch, 'cuda')
            output = pion_model.infer(batch).detach().cpu()
            predicted_ions.append(output)
        predicted_ions = torch.cat(predicted_ions, dim=0).squeeze()
        dark_proteome_predictions[ion] = predicted_ions
        
        row = {"rep_id": rep_id}
        for ion, predictions in dark_proteome_predictions.items():
            row[ion] = sum(predictions > 0.5).item()
        row["total ion sites"] = len(predictions)
    prediction_counts.append(row)
    
    dark_proteome_ion_ids = [x['id'] for x in dark_proteome_ion]
    for ion in chosen_ions:
        for i, pred in enumerate(dark_proteome_predictions[ion]):
            if pred > 0.5:
                rep_ids_predictions[(rep_id, ion)].append(dark_proteome_ion_ids[i])

prediction_counts = pd.DataFrame(prediction_counts)
prediction_counts

A0A496MS18 num Pesto predicted ion binding sites 115
A0A352V9P3 num Pesto predicted ion binding sites 75
W2EKU0 num Pesto predicted ion binding sites 24
A0A0Q8TPY4 num Pesto predicted ion binding sites 28
A0A552UZ29 num Pesto predicted ion binding sites 8


,rep_id,CA,CO,CU,FE,K,MG,MN,NA,ZN,total ion sites
0,A0A496MS18,21,1,0,4,0,27,0,0,0,115
1,A0A352V9P3,0,0,0,0,0,0,0,0,71,75
2,W2EKU0,4,0,0,2,0,5,0,0,0,24
3,A0A0Q8TPY4,0,2,0,0,0,0,0,0,28,28
4,A0A552UZ29,0,0,0,0,0,0,0,0,5,8


In [6]:
import json

# write AF3 format
for rep_id, chosen_ions in rep_ids.items():
    for ion in chosen_ions:
        out_dir = f"/n/netscratch/mzitnik_lab/Lab/afang/AF3-output/MLP_1_vs_other_{ion}_{rep_id}_cluster/"
        os.makedirs(f"{out_dir}/inputs/model", exist_ok=True)
        os.makedirs(f"{out_dir}/outputs/model", exist_ok=True)
        for item_id in rep_ids_predictions[(rep_id, ion)]:
            new_fasta = pdb_to_fasta(f"/n/netscratch/mzitnik_lab/Lab/afang/InteractNN/protein_universe/foldseek_cluster/{rep_id}_cluster/{item_id}.pdb")
            fasta_str = new_fasta[f'{item_id}_A']

            item_job1 = {
                "name": item_id + '_model',
                "modelSeeds": [1],
                "sequences": [
                    {"protein": {"id": 'A', "sequence": fasta_str}},
                    {"ligand": {"id": ['B'], "ccdCodes": [ion]}},
                ],
                "dialect": "alphafold3",
                "version": 1
            }
            with open(f'{out_dir}/inputs/model/{item_id}_model.json', 'w') as f:
                json.dump(item_job1, f, indent=2)

In [5]:
# analyse AF3 output
from glob import glob
import json
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

results_df = []
for rep_id, chosen_ions in rep_ids.items():
    for ion in chosen_ions:
        model_outputs = glob(f"/n/netscratch/mzitnik_lab/Lab/afang/AF3-output/MLP_1_vs_other_{ion}_{rep_id}_cluster/outputs/model/**/*summary_confidences.json")
        for fname in model_outputs:
            with open(fname, 'r') as f:
                data = json.load(f)
            id_name = os.path.basename(fname).replace("_summary_confidences.json", "")
            iptm = data['iptm']
            ptm = data['ptm']
            with open(fname.replace("_summary_confidences.json", "_data.json"), 'r') as f:
                data = json.load(f)
            if 'ccdCodes' in data['sequences'][1]['ligand']:
                ligand = data['sequences'][1]['ligand']['ccdCodes'][0]
            else:
                ligand = data['sequences'][1]['ligand']['smiles']
                if ligand == 'CC=C1C(=C2C=C3C(=CC)C(=C4N3[Fe]56N2C1=Cc7n5c(c(c7C)CCC(=O)O)C=C8N6C(=C4)C(=C8CCC(=O)O)C)C)C':
                    ligand = 'HEC'
            results_df.append({"ligand": ion, "id": id_name.split("_")[0], "type": id_name.split("_")[1], "iptm": iptm, "ptm": ptm, "rep_id": rep_id})
results_df = pd.DataFrame(results_df)

In [6]:
results_df.groupby(['ligand', 'rep_id']).agg({'iptm': ['mean', 'std'], 'ptm': ['mean', 'std']})

iptm                 ptm          
                       mean       std      mean       std
ligand rep_id                                            
CA     A0A496MS18  0.955238  0.013645  0.862381  0.037270
       W2EKU0      0.962500  0.009574  0.790000  0.083267
MG     W2EKU0      0.928000  0.026833  0.850000  0.027386
ZN     A0A0Q8TPY4  0.975357  0.005762  0.882143  0.043064
       A0A352V9P3  0.965200  0.005047  0.883400  0.022095
       A0A552UZ29  0.946000  0.016733  0.884000  0.020736

In [3]:
ion = 'ZN'
rep_id = 'A0A352V9P3'
model_outputs = glob(f"/n/netscratch/mzitnik_lab/Lab/afang/AF3-output/MLP_1_vs_other_{ion}_{rep_id}_cluster/outputs/model/**/*model_model.cif")

cluster_items = []
for cif_file in model_outputs:
    uniprot_id = os.path.basename(cif_file).split("_")[0]
    new_item = process_PL_pdb(cif_file, uniprot_id, "A", "ZN", "B", None, dist_th=8, fragmentation_method='PS_300')[0]
    cluster_items.append(new_item)

print(f"Processed {len(cluster_items)} out of {len(model_outputs)}")

Processed 71 out of 71


In [23]:
from torch_scatter import scatter_mean

structures = glob(f"/n/netscratch/mzitnik_lab/Lab/afang/AF3-output/MLP_1_vs_other_{ion}_{rep_id}_cluster/outputs/model/**/*_model_model.cif")

zinc_coordination_results = defaultdict(list)
for item in cluster_items:
    zn_coords = item['data']['X'][item['data']['A'].index(32)]
    coords = torch.tensor(item['data']['X'][:-2], dtype=torch.float)  # remove global node, and ZN node
    num_segment0 = len(item['data']['segment_ids']) - sum(item['data']['segment_ids'])
    atom_to_block = sum([[i]*block_len for i, block_len in enumerate(item['data']['block_lengths'][:num_segment0])], [])
    atom_to_block = torch.tensor(atom_to_block, dtype=torch.long)
    block_coords = scatter_mean(coords, atom_to_block, dim=0).numpy()
    pairwise_distances = np.linalg.norm(block_coords - zn_coords, axis=1)
    closest_blocks = pairwise_distances.argsort()[1:5]
    # print([pairwise_distances[b] for b in closest_blocks])
    closest_residues = Counter([VOCAB.idx_to_abrv(item['data']['B'][x]) for x in closest_blocks])
    if 'HIS' in closest_residues and closest_residues['HIS'] == 1 and 'CYS' in closest_residues and closest_residues['CYS'] == 3:
        zinc_coordination_results['3CYS1HIS'].append(item['id'])
    elif 'CYS' in closest_residues and closest_residues['CYS'] == 4:
        zinc_coordination_results['4CYS'].append(item['id'])
    else:
        print(item['id'], closest_residues)

for key, val in zinc_coordination_results.items():
    print(key, len(val))

a0a1m6h9f9_A_B_ZN Counter({'CYS': 3, 'PHE': 1})
a0a7u4er34_A_B_ZN Counter({'CYS': 2, 'THR': 1, 'LYS': 1})
a0a849r231_A_B_ZN Counter({'CYS': 2, 'VAL': 1, 'ASN': 1})
3CYS1HIS 35
4CYS 33
